# An autoencoder for pufferfish pictures

Every picture in this notebook shows one pufferfish, and three numbers say everything about it:

* $x$ and $y$, where the fish sits in the frame, as fractions of the frame's width and height measured from its bottom left corner,
* $m$, how puffed up it is, from $0$ (a small flat fish) to $1$ (a big round ball with spikes).

The pictures are synthetic: a few lines of Python below draw them from these three numbers, with a seeded random generator, so every run sees the same fish.

An **autoencoder** is two networks back to back. The **encoder** squeezes a picture of $32 \times 32 = 1024$ pixels into just $3$ numbers, and the **decoder** rebuilds the picture from those $3$ numbers. We train the two together until the rebuilt pictures look like the originals. Nobody tells them about $x$, $y$ and $m$.

In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import ipywidgets as widgets

## 1. Drawing a pufferfish

A picture is a grid of $32 \times 32$ numbers between $0$ (empty water) and $1$ (dark ink). `draw_fish(x, y, m)` paints a light body with a fin on each side, and on top of it a dark face: two eyes and an open mouth. Above $m = 0.5$ the body grows spikes.

In [ ]:
# Where every pixel sits: u runs from 0 at the left edge to 1 at the right edge,
# v from 0 at the bottom edge to 1 at the top edge.
u, v = np.meshgrid(np.linspace(0, 1, 32), np.linspace(1, 0, 32))


def blob(cx, cy, half_width, half_height, spikes=0):
    # An ellipse centred at (cx, cy): 1 inside and 0 outside, fading out over the
    # last quarter of the way to its rim. spikes pushes the rim further out.
    distance = np.sqrt(((u - cx) / half_width) ** 2 + ((v - cy) / half_height) ** 2)
    return np.clip(4 * (1 + spikes - distance), 0, 1)


def draw_fish(x, y, m):
    width = 0.15 + 0.07 * m       # half the width of the body
    height = 0.06 + 0.16 * m      # half its height: flat at m = 0, round at m = 1
    angle = np.arctan2(v - y, u - x)
    spikes = max(0, m - 0.5) * 0.7 * np.cos(7 * angle) ** 8   # 14 spikes, only above m = 0.5
    body = (blob(x, y, width, height, spikes)
            + blob(x - width, y, 0.06, 0.04)                        # left fin
            + blob(x + width, y, 0.06, 0.04))                       # right fin
    face = (blob(x - 0.4 * width, y + 0.3 * height, 0.03, 0.04)     # left eye
            + blob(x + 0.4 * width, y + 0.3 * height, 0.03, 0.04)   # right eye
            + blob(x, y - 0.4 * height, 0.04, 0.03))                # open mouth
    return 0.4 * np.clip(body, 0, 1) + 0.6 * np.clip(face, 0, 1)

## 2. Five thousand fish

We draw $x$, $y$ and $m$ at random and paint one fish for each. The autoencoder will only ever see the pictures; we keep the three numbers behind each one, `truth`, to label the pictures for us.

In [ ]:
rng = np.random.default_rng(seed=0)   # seeded: the same 5000 fish on every computer

# One row (x, y, m) per fish: x and y between 0.25 and 0.75, m between 0 and 1.
truth = rng.uniform(low=[0.25, 0.25, 0], high=[0.75, 0.75, 1], size=(5000, 3))

pictures = np.array([draw_fish(x, y, m) for x, y, m in truth])
pictures = torch.tensor(pictures, dtype=torch.float32)
print(pictures.shape)

In [ ]:
def show(ax, picture, title):
    ax.imshow(picture, cmap="Oranges", vmin=0, vmax=1)
    ax.set(xticks=[], yticks=[], title=title)


fig, axes = plt.subplots(2, 6, figsize=(15, 7))
for i, ax in enumerate(axes.flat):
    x, y, m = truth[i]
    show(ax, pictures[i], f"fish {i}\nx={x:.2f}, y={y:.2f}\nm={m:.2f}")
plt.show()

## 3. The autoencoder

The encoder lays the $32 \times 32$ picture out as one list of $1024$ numbers and passes it through three layers, the last of which has $3$ outputs: the **code** of the picture. The decoder runs the same way backwards, from $3$ numbers to $1024$, one per pixel, which it folds back into a $32 \times 32$ picture. On the slides the encoder is $\varphi$ and the decoder is $\varphi^{-1}$.

In [ ]:
torch.manual_seed(0)   # seeded: the same starting weights on every computer

encoder = nn.Sequential(
    nn.Flatten(),                           # 32 x 32 pixels into one list of 1024 numbers
    nn.Linear(1024, 256), nn.ReLU(),
    nn.Linear(256, 64), nn.ReLU(),
    nn.Linear(64, 3),                       # the code: 3 numbers
)

decoder = nn.Sequential(
    nn.Linear(3, 64), nn.ReLU(),
    nn.Linear(64, 256), nn.ReLU(),
    nn.Linear(256, 1024),                   # 1024 numbers, one per pixel
    nn.Unflatten(1, (32, 32)),              # folded back into 32 x 32 pixels
)

autoencoder = nn.Sequential(encoder, decoder)

## 4. Training

The loop of weekend 1: while the miss is too big, take a step. The miss is the squared difference between a rebuilt pixel and the original one, averaged over all the pixels of $64$ fish picked at random.

In [ ]:
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.001)

loss = 1.0                                                # any start above the threshold
while loss > 0.0018:
    batch = pictures[torch.randint(0, 5000, (64,))]       # 64 fish picked at random
    loss = ((autoencoder(batch) - batch) ** 2).mean()     # the average squared miss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"\raverage squared miss: {loss.item():.4f}", end="")

## 5. Play with the three numbers

Pick a fish with the first slider. On the left is its picture, with the $x$, $y$ and $m$ it was drawn from and the three numbers the encoder turns it into. The three sliders below start at those numbers, and on the right is what the decoder draws from them. Move the sliders and a new fish appears.

Two things to look for:

* The three numbers are the encoder's own. Nobody told it about $x$, $y$ and $m$, so moving one slider may move the fish and puff it up at the same time.
* Every slider stays within the values the encoder gave to some fish, and yet some settings of the three give a smudge instead of a fish: no picture was ever encoded there.

In [ ]:
codes = encoder(pictures).detach()          # the three numbers of every fish
low = codes.min(dim=0).values.tolist()      # the smallest value of each number
high = codes.max(dim=0).values.tolist()     # the largest value of each number

fish = widgets.IntSlider(min=0, max=4999, description="fish", continuous_update=False)
number1 = widgets.FloatSlider(min=low[0], max=high[0], step=0.01, description="number 1")
number2 = widgets.FloatSlider(min=low[1], max=high[1], step=0.01, description="number 2")
number3 = widgets.FloatSlider(min=low[2], max=high[2], step=0.01, description="number 3")


def pick_fish(change):
    # A new fish was picked: move the three sliders to its three numbers.
    number1.value, number2.value, number3.value = codes[fish.value].tolist()


def draw(fish, number1, number2, number3):
    x, y, m = truth[fish]
    a, b, c = codes[fish].tolist()
    new_picture = decoder(torch.tensor([[number1, number2, number3]])).detach()[0]
    fig, (left, right) = plt.subplots(1, 2, figsize=(9, 4.8))
    show(left, pictures[fish],
         f"fish {fish}: x={x:.2f} y={y:.2f} m={m:.2f}\nencoded as {a:.2f}, {b:.2f}, {c:.2f}")
    show(right, new_picture,
         f"decoded from\n{number1:.2f}, {number2:.2f}, {number3:.2f}")
    plt.show()


fish.observe(pick_fish, names="value")
pick_fish(None)   # start on fish 0
widgets.interact(draw, fish=fish, number1=number1, number2=number2, number3=number3);